# Fine-tuning Models for Sentiment Analysis

This notebook demonstrates how to fine-tune a Hugging Face model for sentiment analysis using Amazon SageMaker.

We'll use the GLUE SST-2 dataset to fine-tune a DistilBERT model for binary sentiment classification.

## Setup

First, let's install the required packages:

In [ ]:
!pip install -U pip
!pip install sagemaker boto3 datasets transformers scikit-learn

Import the necessary libraries:

In [ ]:
import os
import json
import logging
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFace
from datasets import load_dataset
from transformers import AutoTokenizer

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

## Configuration

Set up the configuration for the fine-tuning job:

In [ ]:
# Get the default S3 bucket for this SageMaker session
session = sagemaker.Session()
S3_BUCKET = session.default_bucket()
S3_PREFIX = "sentiment-analysis"

print(f"Using S3 bucket: {S3_BUCKET}")

# S3 paths
OUTPUT_PATH = f"s3://{S3_BUCKET}/{S3_PREFIX}/output"

# Model and training parameters
MODEL_ID = "distilbert-base-uncased"
DATASET_NAME = "glue"
DATASET_CONFIG = "sst2"
INSTANCE_TYPE = "ml.g5.xlarge"  # Modern GPU instance with NVIDIA A10G for faster training
INSTANCE_COUNT = 1
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
MAX_LENGTH = 128

## Prepare and Upload Dataset

Let's prepare the SST2 dataset (from GLUE) and convert it to JSON format for SageMaker:

In [ ]:
def prepare_and_save_as_json(dataset_name, dataset_config, model_id, max_length=128):
    """Load, tokenize, and save dataset as JSON files"""
    logger.info(f"Loading dataset: {dataset_name}/{dataset_config}")
    dataset = load_dataset(dataset_name, dataset_config)
    
    logger.info(f"Loading tokenizer for model: {model_id}")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    logger.info("Tokenizing and converting to JSON format")
    os.makedirs("data", exist_ok=True)
    
    # Process training split
    if "train" in dataset:
        train_data = []
        for example in dataset["train"]:
            tokenized = tokenizer(
                example["sentence"],
                padding="max_length",
                truncation=True,
                max_length=max_length,
            )
            train_data.append({
                "input_ids": tokenized["input_ids"],
                "attention_mask": tokenized["attention_mask"],
                "labels": example["label"]
            })
        
        with open("data/train.json", "w") as f:
            json.dump(train_data, f)
        logger.info(f"Saved {len(train_data)} training examples to data/train.json")
    
    # Process validation split
    if "validation" in dataset:
        val_data = []
        for example in dataset["validation"]:
            tokenized = tokenizer(
                example["sentence"],
                padding="max_length",
                truncation=True,
                max_length=max_length,
            )
            val_data.append({
                "input_ids": tokenized["input_ids"],
                "attention_mask": tokenized["attention_mask"],
                "labels": example["label"]
            })
        
        with open("data/validation.json", "w") as f:
            json.dump(val_data, f)
        logger.info(f"Saved {len(val_data)} validation examples to data/validation.json")
    
    return True

In [ ]:
def upload_json_to_s3(s3_bucket, s3_prefix):
    """Upload JSON datasets to S3 using boto3"""
    logger.info("Uploading JSON datasets to S3")
    
    # Initialize S3 client
    s3_client = boto3.client('s3')
    
    s3_paths = {}
    for split_name in ["train", "validation"]:
        local_file = f"data/{split_name}.json"
        if os.path.exists(local_file):
            s3_key = f"{s3_prefix}/data/{split_name}.json"
            s3_path = f"s3://{s3_bucket}/{s3_key}"
            
            # Upload file to S3
            s3_client.upload_file(local_file, s3_bucket, s3_key)
            logger.info(f"Dataset uploaded to {s3_path}")
            s3_paths[split_name] = s3_path
    
    return s3_paths

In [ ]:
# Prepare and upload the dataset
success = prepare_and_save_as_json(DATASET_NAME, DATASET_CONFIG, MODEL_ID, MAX_LENGTH)
if success:
    s3_paths = upload_json_to_s3(S3_BUCKET, S3_PREFIX)
    print(f"Dataset uploaded successfully: {s3_paths}")
else:
    print("Failed to prepare dataset")

## Training Script

The training script is available as a separate file:

In [ ]:
# The training script is available as a separate file: scripts/train.py
# This script contains all the necessary components for fine-tuning:
# - SentimentDataset class for handling JSON data
# - compute_metrics function for evaluation
# - Argument parsing for SageMaker environment variables
# - Main training loop using Hugging Face Trainer

print("Training script: scripts/train.py")

## Fine-tune the Model

Now let's fine-tune the model using SageMaker:

In [ ]:
# Get execution role
role = sagemaker.get_execution_role()
logger.info(f"Using SageMaker execution role: {role}")

In [ ]:
# Define hyperparameters
hyperparameters = {
    'epochs': EPOCHS,
    'train_batch_size': BATCH_SIZE,
    'eval_batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'model_name': MODEL_ID,
}

# Define metric definitions for tracking
metric_definitions = [
    {'Name': 'train:loss', 'Regex': 'train_loss: ([0-9\\.]+)'},
    {'Name': 'eval:loss', 'Regex': 'eval_loss: ([0-9\\.]+)'},
    {'Name': 'eval:accuracy', 'Regex': 'eval_accuracy: ([0-9\\.]+)'},
    {'Name': 'eval:f1', 'Regex': 'eval_f1: ([0-9\\.]+)'},
]

print(f"Hyperparameters: {hyperparameters}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
# Create Hugging Face estimator
logger.info("Creating HuggingFace estimator...")
huggingface_estimator = HuggingFace(
    entry_point='train.py',
    source_dir='./scripts',
    instance_type=INSTANCE_TYPE,
    instance_count=INSTANCE_COUNT,
    role=role,
    transformers_version='4.26.0',
    pytorch_version='1.13.1',
    py_version='py312',
    hyperparameters=hyperparameters,
    metric_definitions=metric_definitions,
    output_path=OUTPUT_PATH
)

print("HuggingFace estimator created successfully!")

In [ ]:
# Define data channels - point to the S3 directories containing the JSON files
train_data_path = s3_paths['train']
validation_data_path = s3_paths['validation']

data_channels = {
    'train': train_data_path.replace('/train.json', ''),
    'validation': validation_data_path.replace('/validation.json', '')
}

print(f"Data channels: {data_channels}")

In [ ]:
# Start training job
logger.info("Starting training job...")
huggingface_estimator.fit(data_channels, wait=True)
logger.info(f"Training job completed. Model artifacts saved to: {huggingface_estimator.model_data}")
print(f"\n✅ Training completed successfully!")
print(f"Model artifacts: {huggingface_estimator.model_data}")

## Deploy the Model (Optional)

Optionally, you can deploy the model to a SageMaker endpoint for real-time inference:

In [ ]:
# Deploy the model (uncomment to deploy)
# predictor = huggingface_estimator.deploy(
#     initial_instance_count=1,
#     instance_type='ml.m5.xlarge'
# )
# print(f"Model deployed to endpoint: {predictor.endpoint_name}")

## Test the Model (Optional)

If you deployed the model, you can test it with some sample text:

In [ ]:
# Test the model (uncomment if you deployed the model)
# sample_texts = [
#     "This movie was fantastic! I really enjoyed it.",
#     "This movie was terrible. I hated it.",
#     "The movie was okay, nothing special."
# ]

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# for text in sample_texts:
#     inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
#     
#     # Convert to list for serialization
#     payload = {
#         "inputs": text
#     }
#     
#     response = predictor.predict(payload)
#     print(f"Text: {text}")
#     print(f"Prediction: {response}")
#     print("-" * 50)

## Clean Up (Optional)

Don't forget to delete the endpoint when you're done to avoid incurring charges:

In [ ]:
# Delete the endpoint (uncomment if you deployed the model)
# predictor.delete_endpoint()
# print("Endpoint deleted successfully!")

## Summary

In this notebook, we demonstrated how to:

1. **Prepare the dataset**: Load the GLUE SST-2 dataset, tokenize it, and convert it to JSON format
2. **Upload to S3**: Store the processed dataset in S3 for SageMaker training
3. **Create a training script**: Write a custom training script using Hugging Face Transformers
4. **Fine-tune the model**: Use SageMaker's HuggingFace estimator to fine-tune DistilBERT
5. **Monitor training**: Track the training job status and metrics
6. **Deploy (optional)**: Deploy the trained model to a SageMaker endpoint
7. **Test (optional)**: Test the deployed model with sample text

The fine-tuned model can now classify text as positive or negative sentiment with improved accuracy on your specific domain.